# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

import re
from PIL import ImageOps
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
import shutil
from PIL import Image

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment

# DATA_DIR   = Path("data")
DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 448

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Preprocess Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

In [ ]:
# ===== text cleaning =====
def clean_text(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = re.sub(r"\s+", " ", x)
    x = x.replace("\u200b", "")
    x = x.replace("\xa0", " ")
    x = x.strip()

    return x

text_cols = [
    "question",
    "hint",
    "lecture",
    "solution",
    "subject",
    "topic",
    "grade",
]

for df in [train_df, val_df, test_df]:
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].apply(clean_text)

print("✅ text cleaning done")

In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    parts = []
    
    # 1. Structured metadata (helps the model understand academic context)
    meta = []
    if pd.notna(row.get("subject")) and str(row["subject"]).strip():
        meta.append(f"Subject: {row['subject']}")
    if pd.notna(row.get("grade")) and str(row["grade"]).strip():
        meta.append(f"Grade: {row['grade']}")
    if pd.notna(row.get("topic")) and str(row["topic"]).strip():
        meta.append(f"Topic: {row['topic']}")
    if meta:
        parts.append("\n".join(meta))

    # 2. Inject contextual information (Hint and Lecture)
    context_parts = []
    if pd.notna(row.get("lecture")) and str(row["lecture"]).strip():
        context_parts.append(f"Background: {str(row['lecture']).strip()}")
    if pd.notna(row.get("hint")) and str(row["hint"]).strip():
        context_parts.append(f"Hint: {str(row['hint']).strip()}")
    if context_parts:
        parts.append("Context:\n" + "\n".join(context_parts))

    # 3. Core question
    parts.append(f"Question: {row['question']}")

    # 4. Format answer options (helps the model align with A, B, C...)
    choices = row["choices"]
    choices_str = "\n".join([f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)])
    parts.append("Choices:\n" + choices_str)

    # 5. Strong instruction constraints (important for boosting accuracy)
    parts.append("Task: Answer the question by selecting the correct letter from the choices above. Output ONLY the letter.")
    parts.append("Answer:")

    # 6. Final combined prompt
    # Ensure the <image> tag appears first — this is the standard input format for SmolVLM
    prompt = "<image>\n" + "\n\n".join(parts)

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"
        
    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))


In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)
    
    def _load_image(self, rel_path: str) -> Image.Image:
        full_path = self.data_dir / "images" / rel_path
        img = Image.open(full_path).convert("RGB")
        img = ImageOps.contain(img, (self.img_size, self.img_size)) 
        img = ImageOps.pad(img, (self.img_size, self.img_size), color=(0, 0, 0))
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df, DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df, DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# ── 3a. Load SmolVLM model + run one inference example ───────────────────────

# ===== Local offline model loading version =====
# LOCAL_MODEL_PATH = "your path"
# MODEL_ID = LOCAL_MODEL_PATH

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = val_ds._load_image(sample["image_path"])
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

In [ ]:
# LoRA / DoRA fine-tuning configuration =====
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,  

    target_modules=["q_proj", "v_proj", "k_proj", "gate_proj", "up_proj", "down_proj"],
    
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    use_dora=True,
)

model = get_peft_model(model, lora_config)

model.enable_input_require_grads()

model.print_trainable_parameters()


In [ ]:
# ===== Fine-tuning training loop with gradient accumulation =====
def collate_batch(batch):
    return {
        "image": [item["image"] for item in batch],
        "text": [item["text"] for item in batch],
        "answer": [item["answer"] for item in batch],
    }

# --- Core training configuration optimizations ---
batch_size = 2             # Reduce per-step input size to prevent OOM at 448 resolution
accumulation_steps = 8     # Accumulate gradients for 8 steps (effective Batch Size = 16) for more stable training
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
num_epochs = 1

# Rebuild DataLoader to match the new batch_size
train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch,
)

model.gradient_checkpointing_enable()
model.train()
torch.cuda.empty_cache()

print(f"start training，Batch Size: {batch_size * accumulation_steps}")

for epoch in range(num_epochs):
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    optimizer.zero_grad() # Initialize gradients
    
    for i, batch in enumerate(pbar):
        try:
            # Prepare inputs
            inputs = processor(
                text=batch["text"],
                images=batch["image"],
                return_tensors="pt",
                padding=True,
                truncation=True,
            )
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            labels = inputs["input_ids"].clone()

            # Forward pass
            outputs = model(**inputs, labels=labels)
            
            # --- Gradient accumulation logic ---
            loss = outputs.loss / accumulation_steps
            loss.backward()

            if (i + 1) % accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
            # ------------------

            pbar.set_postfix({"loss": f"{outputs.loss.item():.4f}"})

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("OOM")
                torch.cuda.empty_cache()
                optimizer.zero_grad() # Clear gradients if an error occurs
                continue
            else:
                raise e

    print(f"Epoch {epoch} completed!")

In [ ]:
# ===== Save fine-tuned model and export ZIP archive =====
OUTPUT_DIR = "/kaggle/working/final_model"

# create directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# save LoRA adapter
model.save_pretrained(OUTPUT_DIR)

# save processor
processor.save_pretrained(OUTPUT_DIR)

print(f"Saved model + processor to: {OUTPUT_DIR}")

# zip
zip_path = "/kaggle/working/final_model_weights.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=OUTPUT_DIR
)

print(f"Zipped to: {zip_path}")

In [ ]:
# ===== Constrained generate inference =====
model.eval()

# valid answer letters
VALID_LETTERS = list(CHOICE_LETTERS)


def predict_answer(row):

    # ===== image =====

    image = Image.open(
        DATA_DIR / "images" / row["image_path"]
    ).convert("RGB")

    image = ImageOps.contain(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = ImageOps.pad(
        image,
        (IMG_SIZE, IMG_SIZE),
        color=(0, 0, 0)
    )

    # ===== prompt =====

    prompt = build_prompt(
        row,
        include_answer=False
    )

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    # ===== generate =====

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    # decode only newly generated token
    generated_text = processor.tokenizer.decode(
        generated_ids[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # ===== extract answer =====

    num_choices = len(row["choices"])

    valid_choices = VALID_LETTERS[:num_choices]

    pred_letter = None

    for c in generated_text:
        if c in valid_choices:
            pred_letter = c
            break

    # fallback
    if pred_letter is None:
        pred_letter = valid_choices[0]

    pred = valid_choices.index(pred_letter)

    return pred


# ===== mini validation benchmark =====

sample_val = val_df.sample(
    50,
    random_state=42
).reset_index(drop=True)

correct = 0

for _, row in tqdm(
    sample_val.iterrows(),
    total=len(sample_val)
):

    pred = predict_answer(row)

    if pred == int(row["answer"]):
        correct += 1

acc = correct / len(sample_val)

print(f"\n✅ Mini-val accuracy: {acc:.4f}")
print(f"Correct: {correct}/{len(sample_val)}")


In [ ]:
# ===== Final submission generation =====
predictions = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):

    pred = predict_answer(row)

    predictions.append({
        "id": row["id"],
        "answer": pred
    })

submission = pd.DataFrame(predictions)

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("submission.csv saved!")